# Assignment 2 - The Bookstore Intelligence Mission
**Name:** YOUR NAME | **Roll No:** YOUR ROLL NO

Website: https://books.toscrape.com/ (sandbox site)

## Mission 1 - Reconnaissance
| Data Needed | HTML Tag / Class Found | Notes |
|---|---|---|
| Book Title | `<a title="...">` inside `<h3>` | Visible text is truncated, so I read the `title` attribute |
| Price | `<p class="price_color">` | Starts with a £ sign, stripped and converted to float |
| Rating | `<p class="star-rating Four">` | Rating is a word in the 2nd class, mapped to 1-5 |
| Availability | `<p class="instock availability">` | Has extra whitespace, cleaned with `strip()` |

Each book sits inside `<article class="product_pod">`; the product link is the `href` of the title anchor.

In [ ]:
!pip install -q requests beautifulsoup4 pandas matplotlib

In [ ]:
import time
import requests
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from urllib.parse import urljoin

STARS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
LISTING = "https://books.toscrape.com/catalogue/page-{}.html"

## Mission 2 - Capture the data (one reusable function)

In [ ]:
def parse_book(card, page_url):
    """Turn one <article class='product_pod'> into a dict."""
    a = card.h3.a
    return {
        "Title": a["title"],
        "Price": float(card.select_one("p.price_color").text.replace("£", "").replace("Â", "")),
        "Rating": STARS[card.select_one("p.star-rating")["class"][1]],
        "Availability": card.select_one("p.availability").get_text(strip=True),
        "URL": urljoin(page_url, a["href"]),
    }

def scrape_page(n, session):
    url = LISTING.format(n)
    r = session.get(url, timeout=15)
    if r.status_code != 200:
        return None
    r.encoding = "utf-8"
    soup = BeautifulSoup(r.text, "html.parser")
    return [parse_book(c, url) for c in soup.select("article.product_pod")]

# test on the first page only
with requests.Session() as s:
    first = scrape_page(1, s)
print(len(first), "books on page 1")
pd.DataFrame(first).head()

## Mission 3 - Scale the operation
`PAGES = 5` (Bonus 1). Set to `50` for the Boss Level - the same loop handles it and stops if a page is missing.

In [ ]:
PAGES = 5
records = []
with requests.Session() as s:
    for n in range(1, PAGES + 1):
        rows = scrape_page(n, s)
        if rows is None:
            print("No page", n, "- stopping")
            break
        records += rows
        print(f"page {n}: {len(rows)} books (total {len(records)})")
        time.sleep(0.5)

df = pd.DataFrame(records)
display(df.head())
print(df.dtypes)
print("missing values:", int(df.isna().sum().sum()), "| duplicates:", int(df.duplicated().sum()))
df.to_csv("competitor_books.csv", index=False)

## Mission 4 - Intelligence analysis

In [ ]:
hi, lo = df.loc[df.Price.idxmax()], df.loc[df.Price.idxmin()]
five = df[df.Rating == 5]
cheap5 = five.loc[five.Price.idxmin()]
pct_stock = df.Availability.str.lower().eq("in stock").mean() * 100

print(f"Q1 Most expensive : {hi.Title} (£{hi.Price:.2f})")
print(f"Q2 Cheapest       : {lo.Title} (£{lo.Price:.2f})")
print(f"Q3 Average price  : £{df.Price.mean():.2f}")
print(f"Q4 5-star books   : {len(five)}")
print(f"Q5 In stock       : {pct_stock:.1f}%")
print("Q6 Top 5 highest-rated (ties broken by lowest price):")
display(df.sort_values(["Rating", "Price"], ascending=[False, True]).head(5)[["Title", "Rating", "Price"]])
print(f"Q7 Cheapest 5-star: {cheap5.Title} (£{cheap5.Price:.2f})")

## Final Challenge - The CEO has £100
This is a 0/1 knapsack problem (maximise total stars, budget £100). I solved it with dynamic programming in pence, so the result is the true optimum rather than a greedy estimate. Ties are broken by lowest cost.

In [ ]:
def best_basket(data, budget_gbp=100):
    cap = int(budget_gbp * 100)
    cost = (data.Price * 100).round().astype(int).tolist()
    val = data.Rating.tolist()
    best = [0] * (cap + 1)
    took = []
    for c, v in zip(cost, val):
        flag = bytearray(cap + 1)
        for b in range(cap, c - 1, -1):
            if best[b - c] + v > best[b]:
                best[b] = best[b - c] + v
                flag[b] = 1
        took.append(flag)
    b = max(range(cap + 1), key=lambda x: (best[x], -x))
    picked = []
    for i in range(len(cost) - 1, -1, -1):
        if took[i][b]:
            picked.append(i)
            b -= cost[i]
    return data.iloc[sorted(picked)]

basket = best_basket(df)
display(basket[["Title", "Price", "Rating"]])
print(f"Books: {len(basket)} | Total cost: £{basket.Price.sum():.2f} | Total rating score: {basket.Rating.sum()}")

## Bonus 3 & 4 - Rating counts and chart

In [ ]:
counts = df.Rating.value_counts().sort_index()
print(counts)
ax = counts.plot(kind="bar", color="steelblue", edgecolor="black")
ax.set_title("Number of books per star rating")
ax.set_xlabel("Star rating"); ax.set_ylabel("Number of books")
plt.xticks(rotation=0); plt.tight_layout()
plt.savefig("rating_counts.png", dpi=150); plt.show()

## Business recommendation (generated from the data)

In [ ]:
avg_all, avg5 = df.Price.mean(), five.Price.mean()
below = five[five.Price < avg_all]
print(f"I would prioritise 5-star books priced below the catalogue average of £{avg_all:.2f}. "
      f"Only {len(five)} of {len(df)} books ({len(five)/len(df)*100:.0f}%) are 5-star, and {len(below)} of them cost less than average "
      f"(5-star average £{avg5:.2f}), so quality does not require a premium price. "
      f"Since {pct_stock:.0f}% of titles are in stock, availability is no differentiator; "
      f"rating-per-pound is, and the £100 basket delivers {basket.Rating.sum()} stars across {len(basket)} books.")